In [52]:
import threading
import pandas as pd

STOCK_URL ='../static/data/'

class BeuData(object):
    _instance_lock = threading.Lock()

    def __new__(cls, *args, **kwargs):
        if not hasattr(cls, '_instance'):
            with BeuData._instance_lock:
                if not hasattr(cls, '_instance'):
                    BeuData._instance = super().__new__(cls)

            return BeuData._instance

        
    def query_stock(self, code, start_date, end_date, frequency, ):
        # 如果code不存在列表内，直接返回
        if not self.__check_code_exist(code=code):
            print('股票代码不存在股票列表内')
            return
        else:
            # 获取股票原始数据
            raw = self.__get_data(code=code,frequency=frequency)
            # 获取复权因子数据
            adjust_factor = self.__get_adjust_factor(code=code)
            # 复权计算
            fore_adjust = self.__adjust_cal(raw=raw, adjust_factor = adjust_factor)
#             back_adjust = self.__adjust_cal(raw=raw, adjust_factor = adjust_factor)
            # 根据start_date与end_date返回数据
    

    def __check_code_exist(self, code):
        # 查询all_stock.csv中是否存在与code匹配
        try:
            code_df = pd.read_csv(STOCK_URL + 'all_stock.csv', usecols=['code'])
            count = code_df.count().code
            if count == 0:
                print('指数代码为空，请检查代码')
                return False
            else:
                is_code_in_list = code in code_df['code'].values
                return is_code_in_list
        except Exception as e:
            raise e
            
            
    def __get_data(self, code, frequency):
        try:
            if frequency == 'd':
                url = 'day/'
            elif frequency == '5':
                if self.__is_code_in_min_ignore(code=code):
                    print(f'{code} 没有分钟交易数据')
                    return
                url ='min/'
            else:
                print('Frequency should be d or 5.')
                
            df = pd.read_csv(STOCK_URL + url + code +'.csv')
            return df
        except Exception as e:
            raise e
            
            
    def __is_code_in_min_ignore(self, code):
        try:
            ignore_df = pd.read_csv(STOCK_URL + 'min_ignore.csv', usecols=['code'])
            is_code_in_min_ignore = code in ignore_df['code'].values
            return is_code_in_min_ignore
        except Exception as e:
            print(e)
            
    def __get_adjust_factor(self, code):
        try:
            adjust_factor = pd.read_csv(STOCK_URL + 'adjust_factor_data.csv')
            condition = adjust_factor['code'] == code
            filter_adjust = adjust_factor[condition]
            return filter_adjust
        except Exception as e:
            print(e)
            
    def __adjust_cal(self, raw, adjust_factor):
#         print(adjust_factor)
        for i in range(len(adjust_factor)):
#             print(adjust_factor.iloc[i].dividOperateDate)
            if i == 0:
                condition = raw['date'] < adjust_factor.iloc[i].dividOperateDate
            else:
                condition = (raw['date'] < adjust_factor.iloc[i].dividOperateDate)&(raw['date'] >= adjust_factor.iloc[i-1].dividOperateDate)
            raw[condition].loc[:,'open':'close'] = raw[condition].loc[:,'open':'close'] * adjust_factor.iloc[i-1].foreAdjustFactor
        print(raw[raw['date']=='2-017-05-24'])
        # TODO Bugfix



beu_data = BeuData()


beu_data.query_stock(code='sh.600000',start_date='1999-01-01', end_date='2014-01-01', frequency='d')
# beu_data.check_code_exist(code='sh.600011')

Empty DataFrame
Columns: [date, code, open, high, low, close, preclose, volume, amount, adjustflag, turn, tradestatus, pctChg, isST]
Index: []


D:\Anacoda\lib\site-packages\pandas\core\indexing.py:965: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[item] = s
